# Delta Lake Demo

Demonstrates core Delta Lake operations:
- Create a Delta table
- Append, update, delete
- MERGE (upsert)
- Time travel
- Schema evolution
- Transaction log inspection

> **Run in**: Azure Databricks, or locally with `pip install pyspark delta-spark`

In [ ]:
import tempfile, os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName('DeltaLakeDemo')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')

DELTA_PATH = os.path.join(tempfile.mkdtemp(), 'orders_delta')
print(f'Delta table path: {DELTA_PATH}')

## 1. Create Delta Table (Version 0)

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

schema = StructType([
    StructField('order_id',    StringType(),  False),
    StructField('customer_id', StringType(),  True),
    StructField('product',     StringType(),  True),
    StructField('quantity',    IntegerType(), True),
    StructField('total',       DoubleType(),  True),
    StructField('status',      StringType(),  True),
])

initial_data = [
    ('ORD001', 'CUST101', 'Widget A', 3, 59.97, 'pending'),
    ('ORD002', 'CUST102', 'Widget B', 1, 49.99, 'pending'),
    ('ORD003', 'CUST103', 'Gadget X', 2, 199.98, 'completed'),
]

df = spark.createDataFrame(initial_data, schema)
df.write.format('delta').mode('overwrite').save(DELTA_PATH)
print('Version 0 written')
spark.read.format('delta').load(DELTA_PATH).show()

## 2. Append New Rows (Version 1)

In [ ]:
new_orders = [
    ('ORD004', 'CUST104', 'Gadget Y', 1, 149.99, 'pending'),
    ('ORD005', 'CUST105', 'Widget C', 4, 139.96, 'completed'),
]
spark.createDataFrame(new_orders, schema).write.format('delta').mode('append').save(DELTA_PATH)
print('Version 1 — after append:')
spark.read.format('delta').load(DELTA_PATH).orderBy('order_id').show()

## 3. MERGE (Upsert) — Version 2

In [ ]:
from delta.tables import DeltaTable

# Simulate status updates arriving from an order-management system
updates = spark.createDataFrame([
    ('ORD001', 'CUST101', 'Widget A', 3, 59.97, 'completed'),  # existing — update status
    ('ORD006', 'CUST106', 'Gadget Z', 2, 299.98, 'pending'),   # new order — insert
], schema)

delta_table = DeltaTable.forPath(spark, DELTA_PATH)
(
    delta_table.alias('target')
    .merge(updates.alias('source'), 'target.order_id = source.order_id')
    .whenMatchedUpdate(set={'status': 'source.status'})
    .whenNotMatchedInsertAll()
    .execute()
)
print('Version 2 — after MERGE:')
spark.read.format('delta').load(DELTA_PATH).orderBy('order_id').show()

## 4. Time Travel

In [ ]:
print('Version 0 (original):')
spark.read.format('delta').option('versionAsOf', 0).load(DELTA_PATH).orderBy('order_id').show()

print('Version 1 (after append):')
spark.read.format('delta').option('versionAsOf', 1).load(DELTA_PATH).orderBy('order_id').show()

## 5. Schema Evolution — Add 'region' Column (Version 3)

In [ ]:
extended_data = spark.createDataFrame([
    ('ORD007', 'CUST107', 'Widget A', 2, 39.98, 'completed', 'EMEA'),
], ['order_id', 'customer_id', 'product', 'quantity', 'total', 'status', 'region'])

(
    extended_data.write
    .format('delta')
    .option('mergeSchema', 'true')  # key option for schema evolution
    .mode('append')
    .save(DELTA_PATH)
)
print('Version 3 — schema evolved with new column:')
spark.read.format('delta').load(DELTA_PATH).orderBy('order_id').show()

## 6. Inspect the Transaction Log

In [ ]:
import json
from pathlib import Path

log_dir = Path(DELTA_PATH) / '_delta_log'
for log_file in sorted(log_dir.glob('*.json')):
    print(f'\n--- {log_file.name} ---')
    with open(log_file) as fh:
        for line in fh:
            entry = json.loads(line)
            # Print only the top-level keys for brevity
            print(f'  Operation keys: {list(entry.keys())}')

## 7. Table History

In [ ]:
delta_table = DeltaTable.forPath(spark, DELTA_PATH)
delta_table.history().select('version', 'timestamp', 'operation', 'operationMetrics').show(truncate=False)